# 🎥 Movie Recommendation System (Collaborative Filtering)

Recommend movies from the **MovieLens** dataset, building up from simple
baselines to collaborative filtering (item-item similarity and matrix
factorization) and a content-based option.

**Workflow:** EDA → per-user split → baselines → item-item CF → matrix
factorization (from scratch) → evaluation (RMSE **and** ranking) → recommend.

The dataset is committed to the repo, so this notebook runs as-is.

## 1. Setup & load

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from sklearn.metrics import mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
ratings = pd.read_csv('data/ratings.csv')
movies = pd.read_csv('data/movies.csv')
title = movies.set_index('movieId')['title']
print(ratings.shape, movies.shape)

## 2. Explore — the sparsity problem

The core challenge of recommendation: the user-movie matrix is ~98% empty, and
movie popularity follows a long tail.

In [ ]:
n_u, n_m = ratings['userId'].nunique(), ratings['movieId'].nunique()
print(f'{n_u} users x {n_m} movies, {len(ratings):,} ratings')
print(f'Sparsity: {(1 - len(ratings)/(n_u*n_m))*100:.2f}% empty')
per_movie = ratings.groupby('movieId').size()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(*zip(*ratings['rating'].value_counts().sort_index().items()), width=0.4, color='#4c72b0')
ax[0].set_title(f"Rating distribution (mean={ratings['rating'].mean():.2f})")
ax[1].hist(per_movie, bins=60, color='#c44e52'); ax[1].set_yscale('log')
ax[1].set_title('Ratings per movie (long tail)'); ax[1].set_xlabel('# ratings')
plt.tight_layout(); plt.show()

## 3. Per-user train/test split

Hold out 20% of each user's ratings — so every user keeps history for the
models and has held-out ratings to score against.

In [ ]:
rng = np.random.RandomState(SEED)
test_idx = []
for _, g in ratings.groupby('userId'):
    k = max(1, int(round(len(g) * 0.2)))
    test_idx += list(rng.choice(g.index.values, size=k, replace=False))
test = ratings.loc[test_idx].reset_index(drop=True)
train = ratings.drop(index=test_idx).reset_index(drop=True)
seen_movies = set(train['movieId'].unique())
test_s = test[test['movieId'].isin(seen_movies)].copy()
y = test_s['rating'].values
print(f'train {len(train):,} / test {len(test):,}')

## 4. Baselines

Global mean, per-user mean, per-movie mean — the bar collaborative filtering
must beat.

In [ ]:
def rmse(a, b): return np.sqrt(mean_squared_error(a, b))
gm = train['rating'].mean()
um = train.groupby('userId')['rating'].mean()
im = train.groupby('movieId')['rating'].mean()
results = {
    'Global mean': rmse(y, np.full(len(y), gm)),
    'User mean': rmse(y, test_s['userId'].map(um).fillna(gm)),
    'Movie mean': rmse(y, test_s['movieId'].map(im).fillna(gm)),
}
print(results)

## 5. Item-item collaborative filtering

Predict a user's rating from how they rated *similar* movies. Similarity is
adjusted-cosine (ratings centered by user mean) over movies with enough ratings.

In [ ]:
MIN_RATINGS, K = 5, 40
counts = train.groupby('movieId').size()
popular = np.sort(counts[counts >= MIN_RATINGS].index.values)
pidx = {m: j for j, m in enumerate(popular)}
uidx = {u: i for i, u in enumerate(np.sort(train['userId'].unique()))}
user_mean = train.groupby('userId')['rating'].mean().to_dict()
item_mean = train.groupby('movieId')['rating'].mean().to_dict()

pr = train[train['movieId'].isin(pidx)]
centered = pr['rating'].values - pr['userId'].map(user_mean).values
Xc = csr_matrix((centered, (pr['userId'].map(uidx), pr['movieId'].map(pidx))),
                shape=(len(uidx), len(popular)))
sim = cosine_similarity(Xc.T.astype(np.float32)); np.fill_diagonal(sim, 0)
user_rated = {u: (g['movieId'].map(pidx).values, g['rating'].values - user_mean[u])
              for u, g in pr.groupby('userId')}

def cf_predict(u, m):
    base = user_mean.get(u, gm)
    if m in pidx and u in user_rated:
        js, res = user_rated[u]; s = sim[pidx[m], js]
        if K < len(s):
            top = np.argpartition(np.abs(s), -K)[-K:]; s, res = s[top], res[top]
        d = np.abs(s).sum()
        if d > 1e-8: return np.clip(base + (s @ res)/d, 0.5, 5)
    return np.clip(item_mean.get(m, base), 0.5, 5)

results['Item-item CF'] = rmse(y, [cf_predict(u, m) for u, m in zip(test_s['userId'], test_s['movieId'])])
print('Item-item CF RMSE:', round(results['Item-item CF'], 4))

## 6. Matrix factorization (from scratch)

Learn latent factors for users and movies so their dot product captures taste.
`r_hat(u,i) = mu + b_u + b_i + p_u · q_i`, trained by SGD. Latent factors
generalize across the sparsity far better than similarity alone.

In [ ]:
users = np.sort(train['userId'].unique()); items = np.sort(train['movieId'].unique())
U = {u: i for i, u in enumerate(users)}; I = {m: j for j, m in enumerate(items)}
u_ar = train['userId'].map(U).values; i_ar = train['movieId'].map(I).values
r_ar = train['rating'].values.astype(float)
F, EPOCHS, LR, REG = 40, 20, 0.005, 0.02
rng = np.random.RandomState(SEED)
mu = r_ar.mean()
bu = np.zeros(len(users)); bi = np.zeros(len(items))
P = rng.normal(0, 0.1, (len(users), F)); Q = rng.normal(0, 0.1, (len(items), F))
order = np.arange(len(r_ar))
for ep in range(EPOCHS):
    rng.shuffle(order)
    for idx in order:
        uu, ii, rr = u_ar[idx], i_ar[idx], r_ar[idx]
        e = rr - (mu + bu[uu] + bi[ii] + P[uu] @ Q[ii])
        bu[uu] += LR*(e - REG*bu[uu]); bi[ii] += LR*(e - REG*bi[ii])
        pu = P[uu].copy()
        P[uu] += LR*(e*Q[ii] - REG*P[uu]); Q[ii] += LR*(e*pu - REG*Q[ii])

def mf_predict(u, m):
    p = mu + (bu[U[u]] if u in U else 0) + (bi[I[m]] if m in I else 0)
    if u in U and m in I: p += P[U[u]] @ Q[I[m]]
    return np.clip(p, 0.5, 5)
results['Matrix Factorization'] = rmse(y, [mf_predict(u, m) for u, m in zip(test_s['userId'], test_s['movieId'])])
print('MF RMSE:', round(results['Matrix Factorization'], 4))

## 7. Compare — rating accuracy

In [ ]:
plt.figure(figsize=(8, 4))
names = list(results); vals = [results[n] for n in names]
plt.bar(names, vals, color=['#bbb','#bbb','#bbb','#4c72b0','#55a868'])
for i, v in enumerate(vals): plt.text(i, v, f'{v:.3f}', ha='center', va='bottom')
plt.ylim(0.8, 1.1); plt.ylabel('Test RMSE'); plt.xticks(rotation=20); plt.title('RMSE by model')
plt.tight_layout(); plt.show()

Matrix factorization wins on RMSE — latent factors beat both the
baselines and item-item similarity.

## 8. Ranking quality — and a cautionary result

RMSE measures rating accuracy, but a recommender is judged by its top-N list.
We measure **precision@10**: of the 10 movies we'd recommend, how many the user
rated ≥4 in the test set.

In [ ]:
def precision_at_k(recommend_fn, k=10):
    train_seen = train.groupby('userId')['movieId'].apply(set)
    relevant = test[test['rating']>=4].groupby('userId')['movieId'].apply(set)
    ps = []
    for user, rel in relevant.items():
        recs = recommend_fn(user, k, train_seen.get(user, set()))
        ps.append(len(set(recs) & rel) / k)
    return np.mean(ps)

pop = train.groupby('movieId').size().sort_values(ascending=False)
def pop_rec(u, n, seen):
    out = [m for m in pop.index if m not in seen][:n]; return out
def mf_rec(u, n, seen):
    if u not in U: return []
    sc = mu + bu[U[u]] + bi + Q @ P[U[u]]
    inv = {j: m for m, j in I.items()}
    out = [inv[j] for j in np.argsort(-sc) if inv[j] not in seen][:n]; return out

print('precision@10  Popularity:', round(precision_at_k(pop_rec), 3))
print('precision@10  MatrixFact:', round(precision_at_k(mf_rec), 3))

**The lesson:** a plain *popularity* baseline often beats personalized models on
offline precision@k — a documented effect called **popularity bias** (the test
set is skewed toward popular movies). So the model that predicts *ratings* best
(MF) isn't automatically the best *ranker*, and popularity is a surprisingly
hard baseline. Recognizing this is central to real recommender work.

## 9. Recommend for a user

In [ ]:
def recommend(u, n=10):
    seen = set(train[train['userId']==u]['movieId'])
    return [(title.get(m, m), float(mu + bu[U[u]] + bi[I[m]] + P[U[u]] @ Q[I[m]]))
            for m in mf_rec(u, n, seen)]

for t, s in recommend(1, 10):
    print(f'  {s:.2f}  {t}')

---
### Next steps
- Optimize directly for ranking (BPR / implicit feedback) instead of RMSE.
- Blend collaborative + content signals (hybrid) to help cold-start items.
- Deploy a `recommend(user)` API behind a small web UI.

*The modular version of this pipeline lives in `src/` — see the README.*